# 🩺 GlicemIA — Notebook de Exploração e Demonstração

Este notebook demonstra o pipeline completo do GlicemIA:
1. Carregamento e análise dos dados CGM
2. Pré-processamento e criação de janelas deslizantes
3. Treinamento do modelo LSTM
4. Visualização de predições vs valores reais
5. Demonstração da classificação de risco


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import MinMaxScaler

# Módulos do projeto
from src.preprocessing.preprocessamento import criar_janelas_deslizantes, gerar_dados_demo

print('✅ Imports carregados com sucesso')

## 1. Dados Sintéticos de Demonstração

Como o dataset real precisa ser baixado separadamente, usamos dados sintéticos que simulam padrões reais de glicemia.

In [ ]:
# Gerar série temporal sintética mais longa para simulação
np.random.seed(42)
n_pontos = 500
t = np.linspace(0, 50, n_pontos)

# Simula padrões fisiológicos: pós-prandial + tendência noturna
glicemia = (
    120                                          # linha de base
    + 60 * np.exp(-0.3 * (t % 8))               # pico pós-prandial
    - 30 * np.sin(2 * np.pi * t / 24)           # variação circadiana
    + np.random.normal(0, 5, n_pontos)           # ruído fisiológico
)
glicemia = np.clip(glicemia, 55, 280)

print(f'Total de leituras CGM simuladas : {n_pontos}')
print(f'Glicemia média                  : {glicemia.mean():.1f} mg/dL')
print(f'Faixa                           : {glicemia.min():.1f} – {glicemia.max():.1f} mg/dL')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Gráfico 1: Série temporal completa
axes[0].plot(glicemia, color='#065A82', linewidth=1.2, label='Glicemia CGM')
axes[0].axhline(70, color='red', linestyle='--', alpha=0.6, label='Hipoglicemia (70)')
axes[0].axhline(90, color='orange', linestyle='--', alpha=0.6, label='Alerta (90)')
axes[0].axhline(180, color='#F96167', linestyle='--', alpha=0.6, label='Hiperglicemia (180)')
axes[0].fill_between(range(n_pontos), 70, 90, alpha=0.08, color='orange', label='Zona de alerta')
axes[0].fill_between(range(n_pontos), 0, 70, alpha=0.08, color='red', label='Zona hipoglicêmica')
axes[0].set_title('Série Temporal CGM Simulada', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Glicemia (mg/dL)')
axes[0].set_xlabel('Leitura (a cada 5 min)')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].grid(alpha=0.3)

# Gráfico 2: Distribuição dos valores
axes[1].hist(glicemia, bins=40, color='#065A82', edgecolor='white', alpha=0.8)
axes[1].axvline(70, color='red', linestyle='--', label='Hipoglicemia (70)')
axes[1].axvline(180, color='#F96167', linestyle='--', label='Hiperglicemia (180)')
axes[1].set_title('Distribuição dos Valores de Glicemia', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Glicemia (mg/dL)')
axes[1].set_ylabel('Frequência')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/eda_glicemia.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em outputs/eda_glicemia.png')

## 2. Pré-processamento: Janelas Deslizantes

In [ ]:
# Normalizar dados
scaler = MinMaxScaler(feature_range=(0, 1))
glicemia_norm = scaler.fit_transform(glicemia.reshape(-1, 1))

# Criar janelas deslizantes
JANELA = 60
HORIZONTE = 60

X, y = criar_janelas_deslizantes(glicemia_norm, JANELA, HORIZONTE)

print(f'Shape X (entradas): {X.shape}  → {X.shape[0]} amostras de {X.shape[1]} leituras')
print(f'Shape y (alvos)   : {y.shape}  → {y.shape[0]} valores alvo')
print(f'\nCada janela representa ~{JANELA * 5 // 60}h de histórico')
print(f'O alvo é a glicemia ~{HORIZONTE * 5} minutos à frente')

In [ ]:
# Visualizar exemplo de janela + alvo
idx_exemplo = 100
janela_ex = scaler.inverse_transform(X[idx_exemplo].reshape(-1, 1)).flatten()
alvo_ex = float(scaler.inverse_transform([[y[idx_exemplo]]])[0][0])

plt.figure(figsize=(12, 4))
x_eixo = list(range(JANELA))
plt.plot(x_eixo, janela_ex, color='#065A82', linewidth=1.5, label='Histórico (entrada)')
plt.scatter([JANELA + HORIZONTE], [alvo_ex], color='#F96167', s=120, zorder=5,
            label=f'Alvo a prever ({alvo_ex:.0f} mg/dL)')
plt.axvline(x=JANELA, color='gray', linestyle=':', alpha=0.6)
plt.annotate(f'  → prever daqui\n     +{HORIZONTE*5}min', xy=(JANELA, janela_ex[-1]),
             fontsize=9, color='gray')
plt.title(f'Exemplo de Janela Deslizante (amostra #{idx_exemplo})', fontsize=13, fontweight='bold')
plt.xlabel('Leitura')
plt.ylabel('Glicemia (mg/dL)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/exemplo_janela.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demonstração da Classificação de Risco

In [ ]:
def classificar_risco(valor_mgdl):
    if valor_mgdl < 70:
        return '🔴 HIPOGLICEMIA', 'red'
    elif valor_mgdl < 90:
        return '🟠 ALERTA', 'orange'
    elif valor_mgdl <= 180:
        return '🟢 NORMAL', 'green'
    else:
        return '🟡 HIPERGLICEMIA', 'goldenrod'

casos_demo = [
    ('Paciente A', 62.0, 'Hipoglicemia severa — ação imediata'),
    ('Paciente B', 83.5, 'Tendência de queda — monitorar'),
    ('Paciente C', 118.0, 'Dentro da faixa normal'),
    ('Paciente D', 215.0, 'Hiperglicemia — verificar dose de insulina'),
]

print(f"{'Paciente':<14} {'Predição':>12}  {'Classificação':<20} {'Contexto'}")
print('─' * 75)
for nome, valor, contexto in casos_demo:
    cls, cor = classificar_risco(valor)
    print(f"{nome:<14} {valor:>10.1f} mg/dL  {cls:<20} {contexto}")

## 4. Dados de Demo para a API

Exemplo de payload pronto para testar o endpoint `POST /predict`

In [ ]:
import json

dados_demo = gerar_dados_demo()

payload_api = {
    'leituras_cgm': [round(float(v), 1) for v in dados_demo],
    'paciente_id': 'demo-001'
}

print('Payload para POST /predict:')
print(json.dumps(payload_api, indent=2)[:500] + '\n  ...')
print(f'\nTotal de leituras: {len(payload_api["leituras_cgm"])}')
print(f'Última leitura   : {payload_api["leituras_cgm"][-1]} mg/dL')
print(f'Tendência        : {"queda ↓" if dados_demo[-1] < dados_demo[0] else "subida ↑"}')

## 5. Arquitetura LSTM — Resumo Visual

In [ ]:
try:
    import tensorflow as tf
    from src.model.treinar import construir_modelo
    model = construir_modelo(tamanho_janela=60)
    model.summary()
except ImportError:
    print('TensorFlow não instalado — resumo do modelo:')
    print()
    arquitetura = [
        ('Input',    '(None, 60, 1)',  '—'),
        ('LSTM-64',  '(None, 60, 64)', '33,280'),
        ('Dropout',  '(None, 60, 64)', '0'),
        ('LSTM-32',  '(None, 32)',     '12,416'),
        ('Dropout',  '(None, 32)',     '0'),
        ('Dense-1',  '(None, 1)',      '33'),
    ]
    print(f"{'Camada':<12} {'Output Shape':<20} {'Parâmetros'}")
    print('─' * 45)
    total = 0
    for nome, shape, params in arquitetura:
        print(f"{nome:<12} {shape:<20} {params}")
        if params != '—':
            total += int(params.replace(',', ''))
    print('─' * 45)
    print(f"{'Total de Parâmetros':<32} {total:,}")